# Notebook 04: Baseline Model Training with Joint CTC-Attention Loss
Trains the Continuous ISL Transformer model (`isl_pose_transformer_base_v1`) on GPU with mixed precision (FP16), gradient accumulation, and checkpoint saving.


In [ ]:
import os, sys, yaml, json, torch
sys.path.insert(0, '.')
from ml.data.dataset import ISLDataset, isl_collate_fn
from ml.models.seq2seq_transformer import ISLSeq2SeqTransformer
from ml.training.trainer import ISLTrainer
from torch.utils.data import DataLoader

with open('ml/configs/baseline_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')


In [ ]:
# Adapt batch size based on available GPU memory
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 15.0:  # A100 / V100
        config['training']['batch_size'] = 32
    elif vram_gb >= 10.0:  # T4
        config['training']['batch_size'] = 16
    else:
        config['training']['batch_size'] = 8
else:
    config['training']['batch_size'] = 4
print(f'Configured batch size: {config["training"]["batch_size"]}')


In [ ]:
# Run Training via CLI or direct trainer invocation
!python ml/training/train.py --epochs 10
print('Training complete! Best checkpoint saved in ml/checkpoints/isl_transformer_best.pt')
